## NN Regression model with selected and full features

In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras import models, layers, regularizers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

In [2]:
df = pd.read_csv("data_train.csv")

In [3]:
selected_features = pd.read_csv('selected_features.csv')
selected_features = selected_features.iloc[:, 0].tolist()
len(selected_features)

35

In [4]:
# Full feature set (drop IDs + targets)
full_feature_cols = [
    c for c in df.columns
    if c not in ["stock_id", "date", "R1M_Usd", "R1M_Usd_C"]
]
len(full_feature_cols)

93

In [5]:
# Training and Validation split in trainging data 2000–2007 for NN tuning
df["date"] = pd.to_datetime(df["date"])

cutoff_date = pd.Timestamp("2006-01-01")

df_train = df[df["date"] <  cutoff_date].copy()   # 2000–2005
df_val   = df[df["date"] >= cutoff_date].copy()   # 2006–2007

print(df_train["date"].min(), df_train["date"].max())
print(df_val["date"].min(),   df_val["date"].max())

1999-12-31 00:00:00 2005-12-31 00:00:00
2006-01-31 00:00:00 2007-12-31 00:00:00


### Running for Selected Features

In [6]:
# Selected-feature matrices for NN regression
X_tr_sel = df_train[selected_features].values
X_val_sel = df_val[selected_features].values

y_tr_reg  = df_train["R1M_Usd"].values
y_val_reg = df_val["R1M_Usd"].values

# For "sign accuracy" classification metric
y_tr_cls  = df_train["R1M_Usd_C"].values
y_val_cls = df_val["R1M_Usd_C"].values

In [7]:
scaler_sel = StandardScaler()
X_tr_sel_scaled  = scaler_sel.fit_transform(X_tr_sel)
X_val_sel_scaled = scaler_sel.transform(X_val_sel)

input_dim_sel = X_tr_sel_scaled.shape[1]
input_dim_sel

35

### NN Model Builder function

In [8]:
def build_nn_reg_model(input_dim, hidden_layers, l2_strength=0.0, dropout_rate=0.0):
    model = models.Sequential()

    # Explicit input layer (preferred API)
    model.add(layers.Input(shape=(input_dim,)))

    for units in hidden_layers:
        model.add(
            layers.Dense(
                units,
                activation="relu",
                kernel_regularizer=regularizers.l2(l2_strength),
            )
        )
        if dropout_rate > 0.0:
            model.add(layers.Dropout(dropout_rate))

    # Regression output
    model.add(layers.Dense(1))

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mse"],
    )
    return model


In [9]:
## Define a few NN configs to experiment on both full and selected feature sets

nn_reg_configs = [
    {
        "name": "nn_small",
        "hidden_layers": [64, 32],
        "l2_strength": 1e-4,
        "dropout": 0.0,
        "epochs": 30,
        "batch_size": 256,
    },
    {
        "name": "nn_medium_dropout",
        "hidden_layers": [128, 64, 32],
        "l2_strength": 1e-4,
        "dropout": 0.2,
        "epochs": 40,
        "batch_size": 256,
    },
    {
        "name": "nn_deep_strong_reg",
        "hidden_layers": [256, 128, 64],
        "l2_strength": 1e-3,
        "dropout": 0.3,
        "epochs": 50,
        "batch_size": 512,
    },
]


In [10]:
nn_reg_results   = []
histories_reg    = {}

## Running experiments on selected features

for cfg in nn_reg_configs:
    print(f"\n=== Training {cfg['name']} ===")
    
    model_reg = build_nn_reg_model(
        input_dim=input_dim_sel,
        hidden_layers=cfg["hidden_layers"],
        l2_strength=cfg["l2_strength"],
        dropout_rate=cfg["dropout"]
    )
    
    history = model_reg.fit(
        X_tr_sel_scaled, y_tr_reg,
        epochs=cfg["epochs"],
        batch_size=cfg["batch_size"],
        validation_data=(X_val_sel_scaled, y_val_reg),
        verbose=0
    )
    
    histories_reg[cfg["name"]] = history
    
    # Predictions on train / val
    y_pred_tr_nn  = model_reg.predict(X_tr_sel_scaled).flatten()
    y_pred_val_nn = model_reg.predict(X_val_sel_scaled).flatten()
    
    # Regression metrics
    mse_tr  = mean_squared_error(y_tr_reg,  y_pred_tr_nn)
    mse_val = mean_squared_error(y_val_reg, y_pred_val_nn)
    r2_tr   = r2_score(y_tr_reg,  y_pred_tr_nn)
    r2_val  = r2_score(y_val_reg, y_pred_val_nn)
    
    # Sign accuracy (classification from regression)
    y_pred_tr_cls  = (y_pred_tr_nn  > 0).astype(int)
    y_pred_val_cls = (y_pred_val_nn > 0).astype(int)
    
    acc_tr  = accuracy_score(y_tr_cls,  y_pred_tr_cls)
    acc_val = accuracy_score(y_val_cls, y_pred_val_cls)
    
    last_val_mse = history.history.get("val_mse", [None])[-1]
    
    nn_reg_results.append({
        "model_name":  cfg["name"],
        "hidden_layers": cfg["hidden_layers"],
        "l2_strength":  cfg["l2_strength"],
        "dropout":      cfg["dropout"],
        "epochs":       cfg["epochs"],
        "batch_size":   cfg["batch_size"],
        "mse_train":    mse_tr,
        "mse_val":      mse_val,
        "r2_train":     r2_tr,
        "r2_val":       r2_val,
        "acc_train":    acc_tr,
        "acc_val":      acc_val,
        "val_mse_last": last_val_mse,
    })


=== Training nn_small ===


2025-11-15 22:05:54.440516: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


896/896 [==============================] - 0s 184us/step

=== Training nn_medium_dropout ===
896/896 [==============================] - 0s 195us/step

=== Training nn_deep_strong_reg ===
896/896 [==============================] - 0s 267us/step


#### Results for Selected Features

In [11]:
nn_reg_results_df = pd.DataFrame(nn_reg_results)
nn_reg_results_df.sort_values("mse_val")

,model_name,hidden_layers,l2_strength,dropout,epochs,batch_size,mse_train,mse_val,r2_train,r2_val,acc_train,acc_val,val_mse_last
1,nn_medium_dropout,"[128, 64, 32]",0.0001,0.2,40,256,0.027400,0.016194,0.012988,-0.013033,0.497070,0.496128,0.016194
2,nn_deep_strong_reg,"[256, 128, 64]",0.0010,0.3,50,512,0.027770,0.016218,-0.000347,-0.014522,0.497070,0.496128,0.016218
0,nn_small,"[64, 32]",0.0001,0.0,30,256,0.026778,0.016341,0.035388,-0.022242,0.512832,0.496965,0.016341


### Running for full features

In [12]:
# Full-feature matrices for NN regression (same 2000–2005 vs 2006–2007 split)
X_tr_full  = df_train[full_feature_cols].values
X_val_full = df_val[full_feature_cols].values

In [13]:
scaler_full = StandardScaler()
X_tr_full_scaled  = scaler_full.fit_transform(X_tr_full)
X_val_full_scaled = scaler_full.transform(X_val_full)

input_dim_full = X_tr_full_scaled.shape[1]
input_dim_full

93

In [14]:
nn_reg_full_results = []
histories_reg_full  = {}

for cfg in nn_reg_configs:
    print(f"\n=== Training (FULL) {cfg['name']} ===")
    
    model_reg_full = build_nn_reg_model(
        input_dim=input_dim_full,
        hidden_layers=cfg["hidden_layers"],
        l2_strength=cfg["l2_strength"],
        dropout_rate=cfg["dropout"]
    )
    
    history_full = model_reg_full.fit(
        X_tr_full_scaled, y_tr_reg,
        epochs=cfg["epochs"],
        batch_size=cfg["batch_size"],
        validation_data=(X_val_full_scaled, y_val_reg),
        verbose=0
    )
    
    histories_reg_full[cfg["name"]] = history_full
    
    # Predictions on train / val
    y_pred_tr_full  = model_reg_full.predict(X_tr_full_scaled).flatten()
    y_pred_val_full = model_reg_full.predict(X_val_full_scaled).flatten()
    
    # Regression metrics
    mse_tr_full  = mean_squared_error(y_tr_reg,  y_pred_tr_full)
    mse_val_full = mean_squared_error(y_val_reg, y_pred_val_full)
    r2_tr_full   = r2_score(y_tr_reg,  y_pred_tr_full)
    r2_val_full  = r2_score(y_val_reg, y_pred_val_full)
    
    # Sign accuracy metrics
    y_pred_tr_full_cls  = (y_pred_tr_full  > 0).astype(int)
    y_pred_val_full_cls = (y_pred_val_full > 0).astype(int)
    
    acc_tr_full  = accuracy_score(y_tr_cls,  y_pred_tr_full_cls)
    acc_val_full = accuracy_score(y_val_cls, y_pred_val_full_cls)
    
    last_val_mse_full = history_full.history.get("val_mse", [None])[-1]
    
    nn_reg_full_results.append({
        "model_name":   cfg["name"],
        "hidden_layers": cfg["hidden_layers"],
        "l2_strength":  cfg["l2_strength"],
        "dropout":      cfg["dropout"],
        "epochs":       cfg["epochs"],
        "batch_size":   cfg["batch_size"],
        "mse_train":    mse_tr_full,
        "mse_val":      mse_val_full,
        "r2_train":     r2_tr_full,
        "r2_val":       r2_val_full,
        "acc_train":    acc_tr_full,
        "acc_val":      acc_val_full,
        "val_mse_last": last_val_mse_full,
    })



=== Training (FULL) nn_small ===
896/896 [==============================] - 0s 182us/step

=== Training (FULL) nn_medium_dropout ===
896/896 [==============================] - 0s 209us/step

=== Training (FULL) nn_deep_strong_reg ===
896/896 [==============================] - 0s 310us/step


In [15]:
nn_reg_full_results_df = pd.DataFrame(nn_reg_full_results)
nn_reg_full_results_df.sort_values("mse_val")

,model_name,hidden_layers,l2_strength,dropout,epochs,batch_size,mse_train,mse_val,r2_train,r2_val,acc_train,acc_val,val_mse_last
2,nn_deep_strong_reg,"[256, 128, 64]",0.0010,0.3,50,512,0.027689,0.016073,0.002592,-0.005453,0.497070,0.496128,0.016073
1,nn_medium_dropout,"[128, 64, 32]",0.0001,0.2,40,256,0.027351,0.016392,0.014736,-0.025408,0.505482,0.497000,0.016392
0,nn_small,"[64, 32]",0.0001,0.0,30,256,0.026258,0.016503,0.054116,-0.032382,0.521965,0.495954,0.016503


In [16]:
# Convert results to DataFrames
nn_reg_results_df      = pd.DataFrame(nn_reg_results)
nn_reg_full_results_df = pd.DataFrame(nn_reg_full_results)

# Add identifier column
nn_reg_results_df["model_type"]      = "NN_Regression_Selected"
nn_reg_full_results_df["model_type"] = "NN_Regression_Full"

# Merge
nn_reg_all = pd.concat(
    [nn_reg_full_results_df, nn_reg_results_df],
    ignore_index=True
)

# Sort (Full block first, then Selected; and within block by mse_val)
nn_reg_all = nn_reg_all.sort_values(
    ["model_type", "mse_val"],
    ascending=[True, True]   # Full first, then selected; and smallest mse first
)

# Reorder columns: model_type first
cols = ["model_type"] + [c for c in nn_reg_all.columns if c != "model_type"]
nn_reg_all = nn_reg_all[cols]

nn_reg_all


,model_type,model_name,hidden_layers,l2_strength,dropout,epochs,batch_size,mse_train,mse_val,r2_train,r2_val,acc_train,acc_val,val_mse_last
2,NN_Regression_Full,nn_deep_strong_reg,"[256, 128, 64]",0.0010,0.3,50,512,0.027689,0.016073,0.002592,-0.005453,0.497070,0.496128,0.016073
1,NN_Regression_Full,nn_medium_dropout,"[128, 64, 32]",0.0001,0.2,40,256,0.027351,0.016392,0.014736,-0.025408,0.505482,0.497000,0.016392
0,NN_Regression_Full,nn_small,"[64, 32]",0.0001,0.0,30,256,0.026258,0.016503,0.054116,-0.032382,0.521965,0.495954,0.016503
4,NN_Regression_Selected,nn_medium_dropout,"[128, 64, 32]",0.0001,0.2,40,256,0.027400,0.016194,0.012988,-0.013033,0.497070,0.496128,0.016194
5,NN_Regression_Selected,nn_deep_strong_reg,"[256, 128, 64]",0.0010,0.3,50,512,0.027770,0.016218,-0.000347,-0.014522,0.497070,0.496128,0.016218
3,NN_Regression_Selected,nn_small,"[64, 32]",0.0001,0.0,30,256,0.026778,0.016341,0.035388,-0.022242,0.512832,0.496965,0.016341


## Backtest - Rebalancing Loop

In [17]:
data = pd.read_csv("data_backtest.csv").copy()
data["date"] = pd.to_datetime(data["date"])
data["ym"]   = data["date"].dt.to_period("M")

In [18]:
backtest_start = "2008-01-01"
backtest_end   = "2018-12-31"

LOOKBACK_MONTHS = [12, 24, 36, 48, 60]   # lookback periods to try

In [19]:
## Function to get month window
def get_month_window(df, current_ym, lookback):
    start_ym = current_ym - lookback    # period math
    mask = (df["ym"] > start_ym) & (df["ym"] < current_ym)
    return df[mask].copy()

In [20]:
## Backtest function using NN regression model

def run_backtest_nn_reg(df, features, lookback_months, cfg):
    """
    Rolling monthly backtest using NN regression model.
    - df: full panel data
    - features: list of feature column names
    - lookback_months: int, number of months in training window
    - cfg: dict with keys: hidden_layers, l2_strength, dropout, epochs, batch_size
    Returns: pd.Series of monthly long-short PnL.
    """

    monthly_returns = []
    month_index = []

    # year-months within backtest period
    months = (
        df.loc[(df["date"] >= backtest_start) & (df["date"] <= backtest_end), "ym"]
          .sort_values()
          .unique()
    )

    for ym in months:
        # 1) Training window: previous L months
        train_df = get_month_window(df, ym, lookback_months)
        # 2) Test month: current ym
        test_df  = df[df["ym"] == ym]

        if len(train_df) < 500 or len(test_df) == 0:
            continue

        # 3) Prepare X, y
        X_train = train_df[features].values.astype("float32")
        y_train = train_df["R1M_Usd"].values.astype("float32")

        X_test  = test_df[features].values.astype("float32")
        y_test  = test_df["R1M_Usd"].values.astype("float32")  # not needed for trading, but fine

        # 4) Scale features (fit on train, transform both)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled  = scaler.transform(X_test)

        # 5) Build & train NN model (fresh each month)
        model = build_nn_reg_model(
            input_dim=X_train_scaled.shape[1],
            hidden_layers=cfg["hidden_layers"],
            l2_strength=cfg["l2_strength"],
            dropout_rate=cfg["dropout"]
        )

        model.fit(
            X_train_scaled,
            y_train,
            epochs=cfg["epochs"],
            batch_size=cfg["batch_size"],
            verbose=0  # no validation here to avoid peeking into test
        )

        # 6) Predict next month
        preds = model.predict(X_test_scaled, verbose=0).flatten()

        # 7) Build long-short portfolio
        test_df = test_df.copy()
        test_df["pred"] = preds
        test_df = test_df.sort_values("pred")

        n = len(test_df)
        if n < 10:
            continue

        long_df  = test_df.iloc[int(0.8 * n):]   # top 20% predicted returns
        short_df = test_df.iloc[:int(0.2 * n)]   # bottom 20% predicted returns

        w_long  = 1.0 / len(long_df)
        w_short = -1.0 / len(short_df)

        long_ret  = (long_df["R1M_Usd"] * w_long).sum()
        short_ret = (short_df["R1M_Usd"] * w_short).sum()

        monthly_returns.append(long_ret + short_ret)
        month_index.append(ym.to_timestamp("M"))  # use month-end date as index

    return pd.Series(monthly_returns, index=month_index)


In [21]:
## BEst config from NN regression experiments

best_nn_cfg =  {
        "name": "nn_deep_strong_reg",
        "hidden_layers": [256, 128, 64],
        "l2_strength": 1e-3,
        "dropout": 0.3,
        "epochs": 50,
        "batch_size": 512,
    }

In [22]:
results_storage_selected = {}

for L in LOOKBACK_MONTHS:
    print(f"Running NN-REG Backtest with Lookback {L} months...")
    pnl_series_nn = run_backtest_nn_reg(data, selected_features, L, best_nn_cfg)
    results_storage_selected[f"NNreg_Lookback_{L}"] = pnl_series_nn

Running NN-REG Backtest with Lookback 12 months...
Running NN-REG Backtest with Lookback 24 months...
Running NN-REG Backtest with Lookback 36 months...
Running NN-REG Backtest with Lookback 48 months...
Running NN-REG Backtest with Lookback 60 months...


In [23]:
results_storage_selected

{'NNreg_Lookback_12': 2008-01-31   -0.002640
 2008-02-29    0.015679
 2008-03-31   -0.019215
 2008-04-30    0.008617
 2008-05-31    0.041343
                 ...   
 2018-07-31    0.136384
 2018-08-31    0.004264
 2018-09-30   -0.010467
 2018-10-31    0.010690
 2018-11-30   -0.037239
 Length: 131, dtype: float64,
 'NNreg_Lookback_24': 2008-01-31   -0.002640
 2008-02-29   -0.009555
 2008-03-31    0.004495
 2008-04-30    0.000534
 2008-05-31    0.015323
                 ...   
 2018-07-31    0.008064
 2018-08-31    0.006096
 2018-09-30    0.000735
 2018-10-31    0.008058
 2018-11-30   -0.014331
 Length: 131, dtype: float64,
 'NNreg_Lookback_36': 2008-01-31   -0.002640
 2008-02-29   -0.009555
 2008-03-31    0.004495
 2008-04-30    0.008617
 2008-05-31    0.015323
                 ...   
 2018-07-31    0.008064
 2018-08-31    0.003897
 2018-09-30    0.005098
 2018-10-31    0.006542
 2018-11-30   -0.001408
 Length: 131, dtype: float64,
 'NNreg_Lookback_48': 2008-01-31   -0.002640
 2008-02-2

In [24]:
results_storage_all = {}

for L in LOOKBACK_MONTHS:
    print(f"Running NN-REG Backtest with Lookback {L} months...")
    pnl_series_nn = run_backtest_nn_reg(data, full_feature_cols, L, best_nn_cfg)
    results_storage_all[f"NNreg_Lookback_{L}"] = pnl_series_nn

Running NN-REG Backtest with Lookback 12 months...
Running NN-REG Backtest with Lookback 24 months...
Running NN-REG Backtest with Lookback 36 months...
Running NN-REG Backtest with Lookback 48 months...
Running NN-REG Backtest with Lookback 60 months...


In [25]:
results_storage_all

{'NNreg_Lookback_12': 2008-01-31   -0.006645
 2008-02-29   -0.018624
 2008-03-31   -0.020093
 2008-04-30    0.005402
 2008-05-31    0.050939
                 ...   
 2018-07-31    0.135926
 2018-08-31   -0.016136
 2018-09-30   -0.007487
 2018-10-31   -0.044481
 2018-11-30   -0.013506
 Length: 131, dtype: float64,
 'NNreg_Lookback_24': 2008-01-31   -0.002640
 2008-02-29   -0.001847
 2008-03-31    0.004495
 2008-04-30    0.012309
 2008-05-31    0.015323
                 ...   
 2018-07-31    0.007403
 2018-08-31    0.004269
 2018-09-30    0.004666
 2018-10-31    0.013094
 2018-11-30   -0.003731
 Length: 131, dtype: float64,
 'NNreg_Lookback_36': 2008-01-31   -0.002640
 2008-02-29   -0.009555
 2008-03-31    0.004495
 2008-04-30    0.008617
 2008-05-31    0.015323
                 ...   
 2018-07-31    0.008064
 2018-08-31   -0.027727
 2018-09-30    0.005098
 2018-10-31   -0.015511
 2018-11-30    0.017173
 Length: 131, dtype: float64,
 'NNreg_Lookback_48': 2008-01-31   -0.002640
 2008-02-2

In [26]:
## Equally Weighted Portfolio

def build_ew_market_series(df):
    # Restrict to backtest period
    df_bt = df[(df["date"] >= backtest_start) & (df["date"] <= backtest_end)].copy()
    df_bt["ym"] = df_bt["date"].dt.to_period("M")

    # Equal-weight average of R1M_Usd across all stocks each month
    ew = df_bt.groupby("ym")["R1M_Usd"].mean()
    ew.index = ew.index.to_timestamp("M")   # month-end timestamps
    return ew

ew_market = build_ew_market_series(data)

In [27]:
## Performace Metrics

def perf_stats(pnl: pd.Series, ew_market: pd.Series):
    """
    pnl: monthly return series of a strategy (or EW itself)
    ew_market: monthly EW market returns
    """
    # Align with EW
    aligned = pd.concat([pnl.astype(float), ew_market.astype(float)], axis=1, join="inner")
    aligned.columns = ["strategy", "ew"]
    r = aligned["strategy"].dropna()
    if len(r) < 2:
        return None

    # Basic stats
    cum = (1 + r).prod()
    ann_ret = cum**(12 / len(r)) - 1

    ann_vol = r.std() * (12**0.5)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else float("nan")

    # Max drawdown
    cum_curve = (1 + r).cumprod()
    peak = cum_curve.cummax()
    dd = (cum_curve / peak) - 1
    max_dd = dd.min()

    # Hit rate
    hit_rate = (r > 0).mean()

    # Correlation to EW
    corr_ew = aligned["strategy"].corr(aligned["ew"])

    return {
        "n_months": len(r),
        "ann_ret": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": max_dd,
        "hit_rate": hit_rate,
        "corr_to_ew": corr_ew,
    }

In [28]:
## Comapring the SELECTED feature set results vs Equally Weighted Portfolio

all_strategies = {}

# NN regressions
for name, series in results_storage_selected.items():
    all_strategies[name] = series

rows = []

# EW market itself
ew_stats = perf_stats(ew_market, ew_market)
ew_stats["strategy_name"] = "EW_Market"
rows.append(ew_stats)

for name, pnl in all_strategies.items():
    stats = perf_stats(pnl, ew_market)
    if stats is None:
        continue
    stats["strategy_name"] = name
    rows.append(stats)

perf_df = pd.DataFrame(rows).set_index("strategy_name")
perf_df = perf_df.sort_values("sharpe", ascending=False)
perf_df

,n_months,ann_ret,ann_vol,sharpe,max_drawdown,hit_rate,corr_to_ew
strategy_name,,,,,,,
NNreg_Lookback_12,131,0.101794,0.118234,0.860951,-0.118145,0.603053,0.100676
EW_Market,131,0.116382,0.201993,0.576170,-0.484271,0.656489,1.000000
NNreg_Lookback_24,131,0.052887,0.095985,0.550986,-0.125511,0.503817,0.284618
NNreg_Lookback_36,131,0.049772,0.101534,0.490199,-0.096675,0.488550,0.330382
NNreg_Lookback_48,131,0.019348,0.059741,0.323863,-0.116134,0.519084,0.093669
NNreg_Lookback_60,131,0.005432,0.038512,0.141046,-0.076076,0.519084,-0.139528


In [29]:
## Comapring the FULL feature set results vs Equally Weighted Portfolio

all_strategies = {}

# NN regressions
for name, series in results_storage_all.items():
    all_strategies[name] = series

rows = []

# EW market itself
ew_stats = perf_stats(ew_market, ew_market)
ew_stats["strategy_name"] = "EW_Market"
rows.append(ew_stats)

for name, pnl in all_strategies.items():
    stats = perf_stats(pnl, ew_market)
    if stats is None:
        continue
    stats["strategy_name"] = name
    rows.append(stats)

perf_df = pd.DataFrame(rows).set_index("strategy_name")
perf_df = perf_df.sort_values("sharpe", ascending=False)
perf_df

,n_months,ann_ret,ann_vol,sharpe,max_drawdown,hit_rate,corr_to_ew
strategy_name,,,,,,,
EW_Market,131,0.116382,0.201993,0.576170,-0.484271,0.656489,1.000000
NNreg_Lookback_48,131,0.053684,0.107705,0.498437,-0.136596,0.526718,0.377428
NNreg_Lookback_36,131,0.055874,0.113569,0.491979,-0.136268,0.473282,0.400362
NNreg_Lookback_24,131,0.043327,0.097987,0.442165,-0.105837,0.519084,0.306041
NNreg_Lookback_12,131,0.035738,0.129059,0.276911,-0.258341,0.511450,-0.009702
NNreg_Lookback_60,131,0.010415,0.048236,0.215910,-0.082721,0.519084,-0.005962


### NN Reg for Full feature set did not perform better than EW Benchmark whereas Selected feature for 12 months lookback performed better than EW Benchmark